# Test 3 — Multi-Seed Robustness — **seed 2718** (optional 4th/5th)

**This file runs seed 42 only.** Its siblings in this folder run the other two. Each needs its own Kaggle session — a cold-start run is ~5-9 h and Kaggle stops at 12 h, so three in one session would die mid-run.

Order: **seed 42 first** (it should reproduce Table 1's 88.2692% and validates the setup), then 123, then 2025.

## What changed from the previous version, and why

| | old (`test_3_multiseed_broken_down/`) | this notebook |
|---|---|---|
| starting weights | our fine-tuned `best_model_text_only.bin` | **`microsoft/graphcodebert-base`** |
| epoch ceiling | 5 | **10**, patience 2 |
| train `code_length` | 384 | **512** |
| eval `code_length` | 384 | 384 |
| test partition | filtered 18,541 | filtered 18,541 |

**The starting-weights change is the important one.** The old notebook loaded the
GraphCodeBERT text-only checkpoint we had already fine-tuned at seed 42, stripped the
classifier, and retrained the head. Every "seed" therefore began from the same converged
encoder that had already seen the whole training set. That measures head-initialisation and
data-order jitter, not fine-tuning seed variance, and it **understates the spread** — in the
direction that makes Table 2's DFG deltas look more trustworthy than the evidence supports.
Since Table 3 exists precisely to bound the noise on those deltas, it has to start cold.

`512` train / `384` eval mirrors `graphcodebert-train-text-only.ipynb` and `test-2` exactly, so
**seed 42 should reproduce Table 1's 88.2692%**. The notebook checks that for you and says so.

## Kaggle setup

- `+ Add Input -> Datasets` -> the `dfgdataset2` corpus
- GPU on, internet on (it downloads `microsoft/graphcodebert-base`)
- No checkpoint input needed — that is the point.

Budget ~5-9 h; the notebook stops at `time_budget_hours` so the session never dies without
writing results.


In [ ]:
# ============================================================
#  SEED 2718  --  FIFTH of five. One seed per Kaggle session.
#  OPTIONAL EXTENSION. Seeds 42/123/2025 are the core three; these
#  last two tighten the variance estimate. See PAPER.md 3.3a for why
#  the target was fixed at five BEFORE any result was seen.
# ============================================================
SEED = 2718
# ============================================================

In [ ]:
!pip install transformers -q

In [ ]:
import os, json, math, random, hashlib, time, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset, SequentialSampler, RandomSampler
from transformers import (AutoTokenizer, RobertaConfig, RobertaModel,
                          get_linear_schedule_with_warmup)
from collections import defaultdict
from tqdm.auto import tqdm

class Args:
    train_file = "/kaggle/input/datasets/hasanmahmudabdullah/dfgdataset2/dataset_graphcodebert.jsonl"
    model_name_or_path = "microsoft/graphcodebert-base"   # COLD START. Not our checkpoint.

    train_code_length = 512      # matches graphcodebert-train-text-only.ipynb
    eval_code_length  = 384      # matches test-2 and the uniform eval window (PAPER.md L3.4)

    train_batch_size = 16
    eval_batch_size  = 32
    learning_rate    = 2e-5
    max_grad_norm    = 1.0
    num_train_epochs = 10
    patience         = 2
    time_budget_hours = 10.5     # Kaggle kills the session at 12h

    test_ratio = 0.10
    val_ratio  = 0.08
    split_seed = 42              # the PARTITION is fixed; only training varies with SEED

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    n_gpu  = torch.cuda.device_count()

args = Args()
args.seed = SEED

print(f"Seed        : {args.seed}")
print(f"Device      : {args.device}  ({args.n_gpu} GPU)")
print(f"Start from  : {args.model_name_or_path}  <- public weights, cold start")
print(f"Train/eval  : {args.train_code_length} / {args.eval_code_length} tokens")
print(f"Ceiling     : {args.num_train_epochs} epochs, patience {args.patience}")

## Split + duplicate filter

Verbatim from `test_scripts/split_and_filter.py`. `infer_source` has **no filename fallback** —
the corpus carries no source key, so every record resolves to `"unknown"` and the split is a
single shuffle. That is what the training notebooks did, and matching them is the whole point;
adding a fallback here rebuilds Partition S and its 89.9% leak (PAPER.md §5.2).

The partition uses `split_seed=42` regardless of `SEED`, so all three runs are scored on the
**same** 18,541 rows. Varying both at once would confound seed variance with partition variance.

In [ ]:
def infer_source(entry):
    for key in ("source", "dataset", "origin", "project"):
        v = entry.get(key)
        if v is not None and str(v).strip() != "":
            return str(v).strip()
    return "unknown"

def allocate_counts(total_needed, groups, fraction):
    raw  = {g: len(v) * fraction for g, v in groups.items()}
    base = {g: int(math.floor(v)) for g, v in raw.items()}
    rem  = total_needed - sum(base.values())
    order = sorted(groups, key=lambda g: (raw[g] - base[g], len(groups[g])), reverse=True)
    for g in order[:rem]:
        base[g] += 1
    return base

def get_split_indices(filepath, test_ratio, val_ratio, seed):
    srcs, hashes, fnames = [], [], []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            e = json.loads(line)
            srcs.append(infer_source(e))
            hashes.append(hashlib.md5(str(e.get('code','')).encode('utf-8','ignore')).hexdigest())
            fnames.append(str(e.get('filename','')).split('_')[0] or 'unknown')
            del e
    total = len(srcs)
    rng = random.Random(seed)
    groups = defaultdict(list)
    for i, s in enumerate(srcs):
        groups[s].append(i)
    for v in groups.values():
        rng.shuffle(v)

    t_target = int(round(total * test_ratio))
    v_target = int(round(total * val_ratio))
    t_alloc = allocate_counts(t_target, groups, test_ratio)
    rest, test_idx = {}, []
    for s, idx in groups.items():
        k = min(t_alloc[s], len(idx))
        test_idx.extend(idx[:k]); rest[s] = idx[k:]
    v_alloc = allocate_counts(v_target, rest, val_ratio / (1.0 - test_ratio))
    val_idx, train_idx = [], []
    for s, idx in rest.items():
        k = min(v_alloc[s], len(idx))
        val_idx.extend(idx[:k]); train_idx.extend(idx[k:])

    train_idx, val_idx, test_idx = sorted(train_idx), sorted(val_idx), sorted(test_idx)
    assert set(train_idx).isdisjoint(test_idx) and set(val_idx).isdisjoint(test_idx)
    print(f"Split: train={len(train_idx):,} val={len(val_idx):,} test={len(test_idx):,}")

    seen = {hashes[i] for i in train_idx}
    seen.update(hashes[i] for i in val_idx)
    before = len(test_idx)
    test_idx = [i for i in test_idx if hashes[i] not in seen]
    print(f"Duplicate filter: dropped {before-len(test_idx):,} "
          f"({(before-len(test_idx))/before:.2%}) -> {len(test_idx):,} clean")
    return train_idx, val_idx, test_idx, fnames

train_idx, val_idx, test_idx, fnames = get_split_indices(
    args.train_file, args.test_ratio, args.val_ratio, args.split_seed)

# Fail closed: every reported number in this project uses these three sizes.
assert (len(train_idx), len(val_idx), len(test_idx)) == (163967, 15997, 18541), (
    f"partition mismatch: {len(train_idx)}/{len(val_idx)}/{len(test_idx)} "
    f"!= 163967/15997/18541 -- results would not be comparable to Tables 1-2")
print("Partition matches test-2/4/6/7 exactly.")

In [ ]:
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.device_count() > 0:
        torch.cuda.manual_seed_all(s)

set_seed(args.seed)

class TextModel(nn.Module):
    def __init__(self, encoder, config):
        super().__init__()
        self.encoder = encoder
        self.config = config
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, 2)

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)[0]
        logits = self.classifier(self.dropout(out[:, 0, :]))
        prob = F.softmax(logits, dim=-1)
        if labels is not None:
            return nn.CrossEntropyLoss()(logits, labels), prob
        return prob

class SimpleCodeDataset(Dataset):
    """max_length is a constructor arg so train (512) and eval (384) windows can differ,
    mirroring how the checkpoints in Tables 1-2 were produced."""
    def __init__(self, tokenizer, file_path, max_length, indices):
        self.tok = tokenizer
        self.max_length = max_length
        with open(file_path, 'r', encoding='utf-8') as f:
            all_lines = f.readlines()
        self.lines = [all_lines[i] for i in indices]
        del all_lines

    def __len__(self):
        return len(self.lines)

    def __getitem__(self, i):
        e = json.loads(self.lines[i])
        enc = self.tok(e.get('code', ''), max_length=self.max_length,
                       truncation=True, padding='max_length', return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'label': torch.tensor(int(e.get('label', 0) or 0), dtype=torch.long)}

print("Loading tokenizer + datasets ...")
tokenizer = AutoTokenizer.from_pretrained(args.model_name_or_path, use_fast=True)
train_ds = SimpleCodeDataset(tokenizer, args.train_file, args.train_code_length, train_idx)
val_ds   = SimpleCodeDataset(tokenizer, args.train_file, args.train_code_length, val_idx)
test_ds  = SimpleCodeDataset(tokenizer, args.train_file, args.eval_code_length,  test_idx)
print(f"  train {len(train_ds):,} | val {len(val_ds):,} | test {len(test_ds):,}")

In [ ]:
@torch.no_grad()
def evaluate(model, ds, desc):
    model.eval()
    loader = DataLoader(ds, sampler=SequentialSampler(ds),
                        batch_size=args.eval_batch_size, num_workers=2)
    probs, labels = [], []
    for batch in tqdm(loader, desc=desc, leave=False):
        p = model(input_ids=batch['input_ids'].to(args.device),
                  attention_mask=batch['attention_mask'].to(args.device))
        probs.extend(p[:, 1].cpu().numpy())
        labels.extend(batch['label'].numpy())
    probs = np.asarray(probs, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int64)
    acc = float(((probs > 0.5).astype(int) == labels).mean())
    return acc, probs, labels

In [ ]:
config = RobertaConfig.from_pretrained(args.model_name_or_path)
config.num_labels = 2
encoder = RobertaModel.from_pretrained(args.model_name_or_path, config=config)
model = TextModel(encoder, config).to(args.device)
print(f"Cold start from {args.model_name_or_path} -- no fine-tuned weights loaded.")

train_loader = DataLoader(train_ds, sampler=RandomSampler(train_ds),
                          batch_size=args.train_batch_size, num_workers=2, drop_last=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=args.learning_rate, eps=1e-8)
# Schedule spans the FULL ceiling, matching the training notebooks. Early stopping may end
# the run before the schedule completes; that is the same behaviour the checkpoints saw.
total_steps = len(train_loader) * args.num_train_epochs
# 10% linear warmup -- graphcodebert-train-text-only.ipynb uses
# num_warmup_steps=int(total_steps * 0.1). An earlier version of this notebook
# passed 0 and seed 42 came back 0.2642pp below Table 1 with a visibly different
# trajectory (peak at epoch 5 rather than 4). Warmup is not cosmetic here.
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)
scaler = torch.cuda.amp.GradScaler()

best_val, best_state, patience_counter, history = -1.0, None, 0, []
t0 = time.time()
stop_reason = "completed all epochs"

for epoch in range(args.num_train_epochs):
    model.train()
    running = 0.0
    for step, batch in enumerate(tqdm(train_loader, desc=f"epoch {epoch+1}/{args.num_train_epochs}")):
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            loss, _ = model(input_ids=batch['input_ids'].to(args.device),
                            attention_mask=batch['attention_mask'].to(args.device),
                            labels=batch['label'].to(args.device))
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), args.max_grad_norm)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        running += loss.item()

    val_acc, _, _ = evaluate(model, val_ds, "val")
    elapsed = (time.time() - t0) / 3600.0
    history.append({'epoch': epoch + 1, 'train_loss': running / max(1, len(train_loader)),
                    'val_acc': val_acc, 'elapsed_h': elapsed})
    print(f"  epoch {epoch+1}: train_loss={running/max(1,len(train_loader)):.4f} "
          f"val_acc={val_acc*100:.4f}%  ({elapsed:.2f}h)")

    if val_acc > best_val:
        best_val, patience_counter = val_acc, 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"    new best")
    else:
        patience_counter += 1
        print(f"    no improvement ({patience_counter}/{args.patience})")
        if patience_counter >= args.patience:
            stop_reason = f"early stopping at epoch {epoch+1}"
            print(f"  -> {stop_reason}")
            break

    if elapsed > args.time_budget_hours:
        stop_reason = f"time budget hit at epoch {epoch+1}"
        print(f"  -> {stop_reason}")
        break

if best_state is not None:
    model.load_state_dict(best_state)
print(f"\nDone: {stop_reason}. Best val accuracy {best_val*100:.4f}%")

In [ ]:
test_acc, test_probs, test_labels = evaluate(model, test_ds, "final test")

def auc(y, p):
    order = np.argsort(p)
    ranks = np.empty(len(p), dtype=np.float64)
    ranks[order] = np.arange(1, len(p) + 1)
    pos, neg = y.sum(), (1 - y).sum()
    return float((ranks[y == 1].sum() - pos * (pos + 1) / 2) / (pos * neg))

tp = int(((test_probs > 0.5) & (test_labels == 1)).sum())
fp = int(((test_probs > 0.5) & (test_labels == 0)).sum())
fn = int(((test_probs <= 0.5) & (test_labels == 1)).sum())
prec = tp / max(1, tp + fp); rec = tp / max(1, tp + fn)
f1 = 2 * prec * rec / max(1e-12, prec + rec)
roc = auc(test_labels, test_probs)

print(f"\nSEED {args.seed}")
print(f"  Accuracy : {test_acc*100:.4f}%")
print(f"  ROC-AUC  : {roc:.4f}")
print(f"  F1       : {f1:.4f}")
print(f"  FN / FP  : {fn:,} / {fp:,}")
print(f"  Test set : {len(test_labels):,}")

if args.seed == 42:
    # ── CONFIG CHECK: epoch-1 validation, NOT final test accuracy ────────────
    # Final test accuracy cannot separate "wrong config" from "same config,
    # different luck". On 2026-09-06 this notebook, with the config since
    # verified correct, landed 0.5123pp from Table 1 through hardware/cuDNN
    # nondeterminism alone: best val differed by 0.205pp and amplified 2.5x on
    # test. The earlier version of this check called that a CONFIG MISMATCH.
    #
    # Epoch 1 runs before that divergence compounds, so it reflects the LR
    # schedule almost purely -- which is what is actually being verified:
    #     correct config (10% warmup) : 86.4725%  -> 0.02pp from the original
    #     wrong config   (no warmup)  : 87.5852%  -> 1.14pp from the original
    # A 50x separation, where test accuracy separated them by 0.51 vs 0.26 --
    # pointing the wrong way. Check the start of the run, not the finish.
    ORIG_EPOCH1_VAL = 86.45   # graphcodebert-train-text-only.ipynb, Table 1's run
    e1 = history[0]['val_acc'] * 100
    d1 = abs(e1 - ORIG_EPOCH1_VAL)
    print("\n  CONFIG CHECK -- epoch-1 validation vs the run that produced Table 1")
    print(f"    original : {ORIG_EPOCH1_VAL:.4f}%")
    print(f"    this run : {e1:.4f}%")
    if d1 < 0.15:
        v = "OK -- LR schedule and data match. Proceed to seeds 123 and 2025."
    elif d1 < 0.50:
        v = "INVESTIGATE -- compare Args against the training notebook before continuing."
    else:
        v = "CONFIG MISMATCH -- do not spend sessions on 123/2025 until resolved."
    print(f"    delta    : {d1:.4f}pp -- {v}")

    print("\n  Context only, NOT a pass/fail -- this number is noisy by design:")
    print(f"    Table 1  : 88.2692%  ROC 0.9571  FN 1,294  FP 881")
    print(f"    this run : {test_acc*100:.4f}%  ROC {roc:.4f}  FN {fn:,}  FP {fp:,}")
    print(f"    delta    : {abs(test_acc*100-88.2692):.4f}pp")
    print(f"    Two runs of the identical procedure have differed by 0.51pp here.")
    print(f"    Read this as a data point on run-to-run spread, not as a verdict.")

In [ ]:
out = {
    'seed': args.seed,
    'accuracy': test_acc, 'roc_auc': roc, 'f1': f1,
    'false_negatives': fn, 'false_positives': fp,
    'test_set_size': int(len(test_labels)),
    'duplicate_filtered': True,
    'best_val_accuracy': best_val,
    'stop_reason': stop_reason,
    'epochs_run': len(history),
    'config': {
        'start_from': args.model_name_or_path,
        'cold_start': True,
        'train_code_length': args.train_code_length,
        'eval_code_length': args.eval_code_length,
        'num_train_epochs': args.num_train_epochs,
        'patience': args.patience,
        'train_batch_size': args.train_batch_size,
        'learning_rate': args.learning_rate,
        'split_seed': args.split_seed,
    },
    'history': history,
}
with open(f'/kaggle/working/test3_seed{args.seed}_results.json', 'w') as f:
    json.dump(out, f, indent=2)

with open(f'/kaggle/working/test3_seed{args.seed}_results.txt', 'w') as f:
    f.write(f"Test 3: Multi-Seed Robustness -- seed {args.seed}\n")
    f.write("=" * 62 + "\n")
    f.write("Model      : GraphCodeBERT text-only\n")
    f.write(f"Start from : {args.model_name_or_path}  (cold start, public weights)\n")
    f.write(f"Training   : {args.train_code_length} tokens, {args.num_train_epochs} epochs / "
            f"patience {args.patience}, lr {args.learning_rate}\n")
    f.write(f"Evaluation : {args.eval_code_length} tokens\n")
    f.write(f"Test set   : {len(test_labels):,} samples, duplicate-filtered\n")
    f.write(f"Stopped    : {stop_reason} (after {len(history)} epochs)\n\n")
    f.write(f"Accuracy   : {test_acc*100:.4f}%\n")
    f.write(f"ROC-AUC    : {roc:.4f}\n")
    f.write(f"F1         : {f1:.4f}\n")
    f.write(f"FN / FP    : {fn:,} / {fp:,}\n\n")
    f.write("Per-epoch validation accuracy:\n")
    for h in history:
        f.write(f"  epoch {h['epoch']:>2}: val_acc={h['val_acc']*100:.4f}%  "
                f"train_loss={h['train_loss']:.4f}  ({h['elapsed_h']:.2f}h)\n")
    f.write("\nProvenance: test_scripts/test-3-multiseed.ipynb\n")

np.save(f'/kaggle/working/test3_seed{args.seed}_probs.npy', test_probs)

print("Saved:")
for s in ('results.json', 'results.txt', 'probs.npy'):
    print(f"  /kaggle/working/test3_seed{args.seed}_{s}")
print("\nAll five seeds done -- run test_scripts/aggregate_test3.py over results/test3/.")